# Talanta panoramas on the Kaggle GPU (internal, private)

One run renders the 8192 x 4096 panorama of each time in TIMES with blender/web_hero/render_panorama.py, from the render pack scene.
Use it when a bake run saved its maps but its panorama failed (10:30 and 18:05 on 2026-10-27).

How to run:
- Settings: Accelerator GPU T4 x2, Internet on, input the private dataset talanta-render-pack.
- "Save Version", then "Save & Run All (Commit)", private. The panoramas stay as the version output (env/<HHMM>/).
- Download the output, then stop. Do not leave an idle session open.


In [ ]:
# Settings of this run
TIMES = ["1030", "1805"]
PRESETS = {"0730": "morning_golden", "1030": "daylight_artistic", "1805": "sunset_artistic", "2000": "night"}
SAMPLES = 128
print(TIMES, SAMPLES)


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /tmp /kaggle/working | tail -2; free -g | head -2

In [ ]:
# Blender and the pack go to /tmp, so that the version output holds the frames only.
import glob, json, os, shutil, subprocess, time
WORK = "/tmp/talanta"
BLENDER_URL = "https://download.blender.org/release/Blender5.2/blender-5.2.2-linux-x64.tar.xz"  # same as the GPU test
BLENDER = f"{WORK}/blender/blender"
t_setup = time.time()
if not os.path.exists(BLENDER):
    os.makedirs(f"{WORK}/blender", exist_ok=True)
    subprocess.run(f"wget -q {BLENDER_URL} -O /tmp/b.tar.xz && tar -xf /tmp/b.tar.xz -C {WORK}/blender --strip-components=1 "
                   "&& rm /tmp/b.tar.xz", shell=True, check=True)
subprocess.run([BLENDER, "-b", "--version"], check=True)

# The pack root holds blender/ and blender/out/frozen_FL.blend (pack 2026-10-26; frozen_FK.blend in pack 7e19362). The input is read-only, so copy it.
PACK = f"{WORK}/pack/talanta_render_pack"
if not os.path.exists(PACK):
    os.makedirs(f"{WORK}/pack", exist_ok=True)
    unpacked = glob.glob("/kaggle/input/**/talanta_render_pack/blender/out/frozen_F*.blend", recursive=True)
    if unpacked:
        shutil.copytree(unpacked[0].rsplit("/blender/", 1)[0], PACK)
    else:
        parts = sorted(q for q in glob.glob("/kaggle/input/**/talanta_render_pack.tar*", recursive=True) if not q.endswith(".sha256"))
        print(parts)
        subprocess.run(f"cat {' '.join(parts)} | tar -x -C {WORK}/pack", shell=True, check=True)
print(os.listdir(PACK))

In [ ]:
# The pack 2026-10-26 (c14cc71) has no app/public/data/manifest.json. render_panorama.py reads features.site_pad_y from it.
# This cell writes the repo copy (commit 1573714) only if the pack lacks the file. A rebuilt pack makes this cell do nothing.
import base64
_manifest = f"{PACK}/app/public/data/manifest.json"
if not os.path.exists(_manifest):
    os.makedirs(os.path.dirname(_manifest), exist_ok=True)
    open(_manifest, "wb").write(base64.b64decode("eyJvcmlnaW5fdXRtIjogWzI0OTU4Ni4zLCA5ODU1NTMxLjI3XSwgImNycyI6ICJFUFNHOjMyNzM3IiwgImdyb3VuZF9lbGV2X20iOiAxODA5LjYsICJvcmlnaW5fbG9ubGF0IjogWzM2Ljc0OTU5NywgLTEuMzA2MDM3XSwgInNvdXJjZXMiOiB7fSwgInRlcnJhaW4iOiBbeyJmaWxlIjogInRlcnJhaW5fc2l0ZS5iaW4iLCAibngiOiAzMDAsICJueiI6IDMwMCwgIngwIjogLTE0OTUuMTY0Njg5OTIwMTczMywgInowIjogLTE1MDAuNzk2NDkxODY1MDY4NywgImR4IjogMTAuMCwgImR6IjogMTAuMCwgIm1pbiI6IC0zOS41ODQxMjE3MDQxMDE1NiwgIm1heCI6IDI5LjYxMDc0NDQ3NjMxODM2fSwgeyJmaWxlIjogInRlcnJhaW5fZmFyLmJpbiIsICJueCI6IDQwMCwgIm56IjogNDAwLCAieDAiOiAtNTk5NS4xNjQ2ODk5MjAxNzMsICJ6MCI6IC02MDAwLjc5NjQ5MTg2NTA2OSwgImR4IjogMzAuMCwgImR6IjogMzAuMCwgIm1pbiI6IC0xMzYuMDM1MzI0MDk2Njc5NywgIm1heCI6IDE0MC41MDcxMjU4NTQ0OTIyfSwgeyJmaWxlIjogInRlcnJhaW5faG9yaXpvbi5iaW4iLCAibngiOiA0MDAsICJueiI6IDQwMCwgIngwIjogLTE5OTk1LjE2NDY4OTkyMDE3MywgInowIjogLTIwMDAwLjc5NjQ5MTg2NTA3LCAiZHgiOiAxMDAuMCwgImR6IjogMTAwLjAsICJtaW4iOiAtMjg2LjAzODgxODM1OTM3NSwgIm1heCI6IDYwOS40ODk4NjgxNjQwNjI1fV0sICJncm91bmQiOiBbeyJmaWxlIjogImdyb3VuZF9zaXRlLmpwZyIsICJ4MCI6IC0xNTAwLjAsICJ6MCI6IC0xNTAwLjAsICJzaXplIjogMzAwMH0sIHsiZmlsZSI6ICJncm91bmRfZmFyLmpwZyIsICJ4MCI6IC02MDAwLjAsICJ6MCI6IC02MDAwLjAsICJzaXplIjogMTIwMDB9LCB7ImZpbGUiOiAiZ3JvdW5kX2hvcml6b24uanBnIiwgIngwIjogLTIwMDAwLjAsICJ6MCI6IC0yMDAwMC4wLCAic2l6ZSI6IDQwMDAwfV0sICJidWlsZGluZ3MiOiB7InBvcyI6ICJidWlsZGluZ3NfcG9zLmJpbiIsICJjb2wiOiAiYnVpbGRpbmdzX2NvbC5iaW4iLCAiaWR4IjogImJ1aWxkaW5nc19pZHguYmluIiwgInNyYyI6ICJidWlsZGluZ3Nfc3JjLmJpbiIsICJzcmNfY29kZXMiOiBbIm9wZW5fYnVpbGRpbmdzXzIwMjMiLCAib3NtX2Zsb29ycyIsICJkZWZhdWx0X2J5X2FyZWEiXSwgImhlaWdodF9zcmMiOiB7Im9wZW5fYnVpbGRpbmdzXzIwMjMiOiA3NTUwLCAib3NtX2Zsb29ycyI6IDcsICJkZWZhdWx0X2J5X2FyZWEiOiA2Njc3fSwgImNvdW50IjogMTQyMzR9LCAicm9hZHMiOiB7InBvcyI6ICJyb2Fkc19wb3MuYmluIiwgImlkeCI6ICJyb2Fkc19pZHguYmluIiwgInciOiAicm9hZHNfdy5iaW4ifSwgInRyZWVzIjogeyJmaWxlIjogInRyZWVzLmJpbiIsICJjb3VudCI6IDU1Mzk4fSwgImZlYXR1cmVzIjogeyJib3dsX291dGVyIjogW1stOTcuNSwgMjcuOF0sIFstOTMuNjcsIDYwLjMyXSwgWy04Ni4yOSwgODIuNDhdLCBbLTc0Ljg3LCAxMDAuMV0sIFstNjEuNDEsIDEwOS40MV0sIFstNDIuNjQsIDExOS40OF0sIFstMTUuMjQsIDEyNS41XSwgWzIzLjMxLCAxMjQuNDZdLCBbNDguMTYsIDEyMC40MV0sIFs2OC40NCwgMTEzLjA4XSwgWzc4LjMyLCAxMDcuMDJdLCBbODguNzEsIDk2LjE4XSwgWzk4LjM0LCA4MC44XSwgWzEwMy44NywgMjkuODldLCBbMTA1LjYzLCA3LjIxXSwgWzEwNS44NiwgLTE2LjQ3XSwgWzEwMi41MiwgLTYwLjMyXSwgWzk5Ljk3LCAtNzMuMTZdLCBbOTUuMTQsIC04Ni43Nl0sIFs4MS4xOCwgLTEwMy42NF0sIFs2MS44OSwgLTExNi43M10sIFs0Mi42MSwgLTEyNS4yN10sIFsyMi4zNywgLTEyOS42OF0sIFswLjUxLCAtMTMwLjc4XSwgWy0yNi4xMiwgLTEyNy40OF0sIFstNjAuMSwgLTExNy4xMl0sIFstODQuNjcsIC05MC4xM10sIFstOTEuNSwgLTc2LjAyXSwgWy05Ni4wNSwgLTU4LjYyXSwgWy05Ny41NCwgLTE3LjNdLCBbLTk3LjUsIDI3LjhdXSwgImJvd2xfaG9sZSI6IFtbLTUzLjg2LCA0Ny45M10sIFstNTUuOTMsIDQuMzRdLCBbLTUzLjkzLCAtMzMuNzJdLCBbLTUxLjkzLCAtNTcuOTFdLCBbLTQ4LjY0LCAtNjYuOTldLCBbLTQxLjgsIC03NC4wNV0sIFstMjguMzYsIC03OC4zNF0sIFstMTMuOTEsIC04MC44OF0sIFszLjA5LCAtODEuOV0sIFszNS44LCAtNzkuNDFdLCBbNDUuMTksIC03NS44OV0sIFs1My41NywgLTY5LjA5XSwgWzU4LjE1LCAtNTUuNDldLCBbNTguNjgsIC0yMi40OF0sIFs1OS4yMiwgNC45OF0sIFs1OS4xLCAxMC4yOF0sIFs1NS4xNiwgMTAuMTVdLCBbNTQuNjcsIDMwLjddLCBbNTEuNjUsIDU5LjkyXSwgWzQ4LjM2LCA2Ny4yNF0sIFs0MS41MiwgNzMuMDRdLCBbMjEuOTksIDc2Ljg0XSwgWzEuNzEsIDc4LjEyXSwgWy0zOS42MywgNzYuMTNdLCBbLTQ4Ljc3LCA2Ny4wN10sIFstNTMuODYsIDQ3LjkzXV0sICJwaXRjaF96b25lIjogW1stMzUuMzcsIC01OC4zMV0sIFstMzguOTEsIDU2LjkyXSwgWzM2LjgxLCA1Ny4wN10sIFszNy43NSwgLTU3LjBdLCBbLTM1LjM3LCAtNTguMzFdXSwgInBpdGNoX2JlYXJpbmdfZGVnIjogMS43NiwgInNpdGUiOiBbWy0zNzcuOTcsIC0xMzcuMjhdLCBbLTMwNC41NywgNjUuNjZdLCBbLTE4NS45LCAzMjEuOV0sIFs0ODQuNTksIDEwMi40N10sIFsxODcuNjEsIC0zNDkuMjldLCBbLTY1Ljk4LCAtMjU1LjQ5XSwgWy0xNjYuMjEsIC0yMjcuMjhdLCBbLTI1OS4xMSwgLTE4Ni44NV0sIFstMzc3Ljk3LCAtMTM3LjI4XV0sICJzcG9ydCI6IFt7ImtpbmQiOiAicGl0Y2giLCAicmluZyI6IFtbMTM1Ljg0LCAtMTY1LjU3XSwgWzEzNi40OCwgLTY2LjE5XSwgWzE0MC45LCAtNTcuNDRdLCBbMTQ4LjQ4LCAtNDguNTddLCBbMTU0LjY4LCAtNDMuNzhdLCBbMTYxLjk5LCAtNDEuNDZdLCBbMTcxLjYyLCAtNDAuMjRdLCBbMTc5LjE5LCAtNDEuMDddLCBbMTg4LjQzLCAtNDQuMzZdLCBbMTk1LjI5LCAtNDkuM10sIFsyMDAuOTQsIC01NC42Ml0sIFsyMDQuMjUsIC01OS42OV0sIFsyMDYuNzIsIC02NC43NV0sIFsyMDcuOTYsIC02OS42OF0sIFsyMDguOTYsIC0xNjQuNDddLCBbMjA3LjE3LCAtMTcxLjQ0XSwgWzIwMi4zNCwgLTE3OC41Nl0sIFsxOTMuNjYsIC0xODcuNDNdLCBbMTgzLjg5LCAtMTkxLjhdLCBbMTczLjgyLCAtMTk0LjEyXSwgWzE2NS40MywgLTE5My43MV0sIFsxNTYuNzYsIC0xOTAuNTVdLCBbMTQ4LjUsIC0xODQuNjZdLCBbMTQyLjE3LCAtMTc4Ljc3XSwgWzEzOC4xOCwgLTE3My4xN10sIFsxMzUuODQsIC0xNjUuNTddXSwgImhvbGVzIjogW10sICJhcmVhIjogMTAwMjB9LCB7ImtpbmQiOiAidHJhY2siLCAicmluZyI6IFtbMTMzLjcyLCAtMTgxLjE2XSwgWzEzMy43LCAtMTgxLjA5XSwgWzEyNC45OSwgLTE4MS4wM10sIFsxMjUuNzksIC01OC4wN10sIFsxMzIuNjMsIC01OC4xMV0sIFsxMzYuODEsIC00OS44M10sIFsxNDUuNCwgLTM5Ljc0XSwgWzE1Mi40NCwgLTM0LjMyXSwgWzE2MC43NCwgLTMxLjY4XSwgWzE3MS42NywgLTMwLjI5XSwgWzE4MC4yNywgLTMxLjI0XSwgWzE5MC43NCwgLTM0Ljk3XSwgWzE5OC41NCwgLTQwLjU3XSwgWzIwNC45NSwgLTQ2LjYzXSwgWzIwOC42OSwgLTUyLjM3XSwgWzIxMS41MSwgLTU4LjExXSwgWzIxMi45LCAtNjMuNzFdLCBbMjE0LjA1LCAtMTcxLjI5XSwgWzIxMi4wMiwgLTE3OS4yMV0sIFsyMDYuNTQsIC0xODcuMjhdLCBbMTk2LjY5LCAtMTk3LjM2XSwgWzE4NS41OSwgLTIwMi4zMV0sIFsxNzQuMTcsIC0yMDQuOTRdLCBbMTY0LjY0LCAtMjA0LjQ3XSwgWzE1NC44LCAtMjAwLjg5XSwgWzE0NS40MywgLTE5NC4yXSwgWzEzOC4yNSwgLTE4Ny41M10sIFsxMzMuNzIsIC0xODEuMTZdXSwgImhvbGVzIjogW1tbMTM4LjE4LCAtMTczLjE3XSwgWzE0Mi4xNywgLTE3OC43N10sIFsxNDguNSwgLTE4NC42Nl0sIFsxNTYuNzYsIC0xOTAuNTVdLCBbMTY1LjQzLCAtMTkzLjcxXSwgWzE3My44MiwgLTE5NC4xMl0sIFsxODMuODksIC0xOTEuOF0sIFsxOTMuNjYsIC0xODcuNDNdLCBbMjAyLjM0LCAtMTc4LjU2XSwgWzIwNy4xNywgLTE3MS40NF0sIFsyMDguOTYsIC0xNjQuNDddLCBbMjA3Ljk2LCAtNjkuNjhdLCBbMjA2LjcyLCAtNjQuNzVdLCBbMjA0LjI1LCAtNTkuNjldLCBbMjAwLjk0LCAtNTQuNjJdLCBbMTk1LjI5LCAtNDkuM10sIFsxODguNDMsIC00NC4zNl0sIFsxNzkuMTksIC00MS4wN10sIFsxNzEuNjIsIC00MC4yNF0sIFsxNjEuOTksIC00MS40Nl0sIFsxNTQuNjgsIC00My43OF0sIFsxNDguNDgsIC00OC41N10sIFsxNDAuOSwgLTU3LjQ0XSwgWzEzNi40OCwgLTY2LjE5XSwgWzEzNS44NCwgLTE2NS41N10sIFsxMzguMTgsIC0xNzMuMTddXV0sICJhcmVhIjogMzYzOH0sIHsia2luZCI6ICJwaXRjaCIsICJyaW5nIjogW1sxNDkuNTMsIDMxLjExXSwgWzE1MS4xNiwgMTU2LjA5XSwgWzIyNi4yMywgMTU3LjU0XSwgWzIyNy42NCwgMzIuMDZdLCBbMTQ5LjUzLCAzMS4xMV1dLCAiaG9sZXMiOiBbXSwgImFyZWEiOiA5NTkyfV0sICJzaXRlX3BhZF95IjogNi4xMn0sICJsYWJlbHMiOiBbeyJ0ZXh0IjogIkNlbWV0ZXJ5IFJvYWQiLCAieCI6IDI4Mi4yLCAieiI6IDQxOS4yLCAieSI6IDIuNDEyNTY4NzA4NTM3OTQ3LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJDaXR5IE9mIE5haXJvYmkgRHJpdmUiLCAieCI6IDE0ODIuMywgInoiOiA4NC4zLCAieSI6IC04LjkwMDI5NTg1NjYyMjI3OSwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiSGF0aGVydSBSb2FkIiwgIngiOiA5NTMuNSwgInoiOiAtMTU4NS4zLCAieSI6IC0xMS43NzUxNzI2MjMzNjc3OCwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiSG9tZXdhcmQgQm91bmQiLCAieCI6IC0yODguMiwgInoiOiAxNjU0LjksICJ5IjogMTAuMTk4MjQ0MDkyNzU4NjE0LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJKYW1odXJpIFBhcmsgUm9hZCIsICJ4IjogNDU2LjksICJ6IjogLTExOS40LCAieSI6IC04LjEyNjI4MTc3NjMyOTUwNiwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiSmFtdWh1cmkgUm9hZCIsICJ4IjogNTg1LjAsICJ6IjogMzI2LjcsICJ5IjogLTAuOTMyNTM3MzIwNDA0NDM1NCwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiS2FiaXJpYSBSb2FkIiwgIngiOiAtMTQ0Ni43LCAieiI6IC0xNDY3LjgsICJ5IjogMTMuODU2OTA4MTA0MDU3MzExLCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJLYXJhbmRpbmkgUm9hZCIsICJ4IjogODMyLjMsICJ6IjogLTY5Mi41LCAieSI6IC0zLjY1NTk1MTc0NTM4NTIwOTUsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIktlbnlhdHRhIERyaXZlIiwgIngiOiAxNTk0LjAsICJ6IjogMTYyLjAsICJ5IjogLTkuNzY4MDc2MjQyMTQ3MTE3LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJLaW5nYXJhIENsb3NlIiwgIngiOiAxMjEzLjIsICJ6IjogLTExMzcuMywgInkiOiAtMjguNTA2OTc3OTcyOTg3NjcsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIktpbmdhcmEgUm9hZCIsICJ4IjogMTMwOS4xLCAieiI6IC0xMTM1LjUsICJ5IjogLTMxLjI2NDMzODk3MzE2MjY5NCwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiS2lueWFuanVpIENyZXNjZW50IiwgIngiOiAtODY3LjAsICJ6IjogLTEzNzIuMywgInkiOiA2LjIxODg3NDY0NDIwMjk3LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJLaW55YW5qdWkgUm9hZCIsICJ4IjogLTg1OC4xLCAieiI6IC0xMjE3LjQsICJ5IjogNS44NzI0NjI4OTMzMzY2NiwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTWFjaGFyaWEgUm9hZCIsICJ4IjogMjk3LjQsICJ6IjogLTE2NjEuNSwgInkiOiAtMjAuNjgyMjA3OTg4MjY4Mzc0LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJNYW1hIFdhaHUgUm9hZCIsICJ4IjogLTEwNy43LCAieiI6IC01NzguOSwgInkiOiA1LjMwMjA2MjExNDY2ODE2MSwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTWJhYXppIEF2ZW51ZSIsICJ4IjogMTQ3NC4yLCAieiI6IC0xNDgzLjIsICJ5IjogLTEwLjc1ODMzNzA1MzQ1MzcxNSwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTWVtb3JpYWwiLCAieCI6IC0zMDcuNywgInoiOiA4Mi42LCAieSI6IDguMjQzNjg4NDY3OTEzMjE2LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJNb2kgRHJpdmUiLCAieCI6IDE1MjYuMSwgInoiOiAzMDYuOCwgInkiOiAtMTEuNTY4MjUxNzM4MTE3NjI0LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJNdC4gRWxnb24gRHJpdmUiLCAieCI6IDE1NDUuOCwgInoiOiAzNjcuNCwgInkiOiAtMTEuNzI4ODI3MjczOTU5OTUsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIk5haXZhc2hhIFJvYWQiLCAieCI6IDQxOS4zLCAieiI6IC0xMzk5LjcsICJ5IjogLTIyLjMxMzA0Mzc3ODAxNDg0NCwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTmdvbmcgUm9hZCIsICJ4IjogLTYwMS41LCAieiI6IC05Mi4zLCAieSI6IDkuNzEwNTI2ODQ0NTQyMywgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTmdvdGhvIFJvYWQiLCAieCI6IC01NDcuMiwgInoiOiAtMTM0My45LCAieSI6IC02LjgwODM1OTgyNjQyMDg4MywgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiTnlha2lueXVhIFJvYWQiLCAieCI6IC01NTguMywgInoiOiAtMTU3OC41LCAieSI6IC02Ljg4Njg5OTczMjIwNDQ4MywgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiT2F1IFJvYWQiLCAieCI6IDEzODAuMywgInoiOiAxLjIsICJ5IjogLTEzLjE1ODYzNjIzNTg4NDYxNSwgImtpbmQiOiAicm9hZCJ9LCB7InRleHQiOiAiUmlhcmEgUm9hZCIsICJ4IjogMTI0NC4zLCAieiI6IC05OTEuMCwgInkiOiAtMTIuMjIyMDE3NTA2ODcwMDg4LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJTb3V0aGVybiBCeXBhc3MiLCAieCI6IC0xNzI3LjYsICJ6IjogMTQxMi4wLCAieSI6IDMwLjQ4OTEyNDE3NDc5NTk2LCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJTdC4gRGFuaWVsIENvbWJvbmkgUm9hZCIsICJ4IjogNDgxLjIsICJ6IjogLTY5NS42LCAieSI6IC0xLjI4NjI1MDI3MTkyOTAyMDUsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIldhbWJpcmkgUm9hZCIsICJ4IjogLTEzMjEuNSwgInoiOiAtMTc0LjMsICJ5IjogMTYuMzE1MDE5NjIyMTUyMjUsICJraW5kIjogInJvYWQifSwgeyJ0ZXh0IjogIldhbnllZSBSb2FkIiwgIngiOiAyMzMuNywgInoiOiAtODIwLjYsICJ5IjogMC43MDQ1MTcyMDE2MTc1OTExLCAia2luZCI6ICJyb2FkIn0sIHsidGV4dCI6ICJOZ29uZyBSb2FkIEZvcmVzdCBTYW5jdHVhcnkiLCAieCI6IC04NDUsICJ6IjogNjYwLCAieSI6IDIzLCAia2luZCI6ICJhcmVhIiwgImVsZXYiOiAxODE3fSwgeyJ0ZXh0IjogIk5nb25nIFJhY2Vjb3Vyc2UiLCAieCI6IC0xNzM2LCAieiI6IDYwNiwgInkiOiA0MSwgImtpbmQiOiAiYXJlYSIsICJlbGV2IjogMTgzNn0sIHsidGV4dCI6ICJLaWJlcmEiLCAieCI6IDM3MjAsICJ6IjogNzY3LCAieSI6IC01OSwgImtpbmQiOiAiYXJlYSIsICJlbGV2IjogMTczNX0sIHsidGV4dCI6ICJKYW1odXJpIFBhcmsiLCAieCI6IDE3MTUsICJ6IjogMTYwLCAieSI6IC0yLCAia2luZCI6ICJhcmVhIiwgImVsZXYiOiAxNzkyfSwgeyJ0ZXh0IjogIktpbGltYW5pIiwgIngiOiAzODI5LCAieiI6IC0xODg4LCAieSI6IC0zNCwgImtpbmQiOiAiYXJlYSIsICJlbGV2IjogMTc2MX0sIHsidGV4dCI6ICJOYWlyb2JpIENCRCIsICJ4IjogODA0OCwgInoiOiAtMjE3OSwgInkiOiAtMTIxLCAia2luZCI6ICJmYXIiLCAiZWxldiI6IDE2NzR9LCB7InRleHQiOiAiTmdvbmcgSGlsbHMgXHUwMGI3IExhbXdpYSIsICJ4IjogLTEyMzQ2LCAieiI6IDEyNjUyLCAieSI6IDY0NiwgImtpbmQiOiAicGVhayIsICJlbGV2IjogMjQ0MX0sIHsidGV4dCI6ICJOZ29uZyBIaWxscyIsICJ4IjogLTEyNDQ4LCAieiI6IDEwMDUyLCAieSI6IDU4OCwgImtpbmQiOiAicGVhayIsICJlbGV2IjogMjM4M30sIHsidGV4dCI6ICJOZ29uZyIsICJ4IjogLTk3NDgsICJ6IjogNTMxNSwgInkiOiAxMzQsICJraW5kIjogImZhciIsICJlbGV2IjogMTkyOX0sIHsidGV4dCI6ICJXaWxzb24gQWlycG9ydCIsICJ4IjogNzI2MSwgInoiOiAxNzI2LCAieSI6IC0xMTcsICJraW5kIjogImZhciIsICJlbGV2IjogMTY3N31dLCAidGltZWxpbmUiOiBbeyJkYXRlIjogIjIwMjMtMDEtMDgiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMCwgInJpbmdfbWVhbl92aXMiOiA4MDd9LCB7ImRhdGUiOiAiMjAyMy0wMS0xMyIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDkzM30sIHsiZGF0ZSI6ICIyMDIzLTAxLTE4IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTA0MH0sIHsiZGF0ZSI6ICIyMDIzLTAxLTI4IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTE0N30sIHsiZGF0ZSI6ICIyMDIzLTAyLTA3IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTM3OH0sIHsiZGF0ZSI6ICIyMDIzLTAyLTEyIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTQ3OX0sIHsiZGF0ZSI6ICIyMDIzLTAyLTE3IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTYzMX0sIHsiZGF0ZSI6ICIyMDIzLTAyLTIyIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTU3OH0sIHsiZGF0ZSI6ICIyMDIzLTAzLTA5IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTUxOX0sIHsiZGF0ZSI6ICIyMDIzLTA4LTIxIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTE1NX0sIHsiZGF0ZSI6ICIyMDIzLTA4LTI2IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTAwNH0sIHsiZGF0ZSI6ICIyMDIzLTA5LTI1IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogODY5fSwgeyJkYXRlIjogIjIwMjMtMTEtMDkiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMCwgInJpbmdfbWVhbl92aXMiOiA3MTB9LCB7ImRhdGUiOiAiMjAyMy0xMS0yOSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDY0M30sIHsiZGF0ZSI6ICIyMDI0LTAxLTIzIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTAzM30sIHsiZGF0ZSI6ICIyMDI0LTAyLTE3IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjE5OCwgInJpbmdfbWVhbl92aXMiOiAxNTk2fSwgeyJkYXRlIjogIjIwMjQtMDMtMDMiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMCwgInJpbmdfbWVhbl92aXMiOiA4ODZ9LCB7ImRhdGUiOiAiMjAyNC0wMy0xMyIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyNzh9LCB7ImRhdGUiOiAiMjAyNC0wNC0yMiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyODB9LCB7ImRhdGUiOiAiMjAyNC0wNS0yNyIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEzMjJ9LCB7ImRhdGUiOiAiMjAyNC0wOC0wNSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEwNjl9LCB7ImRhdGUiOiAiMjAyNC0wOS0wOSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDExOTB9LCB7ImRhdGUiOiAiMjAyNC0wOS0xNCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyNDl9LCB7ImRhdGUiOiAiMjAyNC0wOS0xOSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyMzR9LCB7ImRhdGUiOiAiMjAyNC0xMC0wNCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wMjQsICJyaW5nX21lYW5fdmlzIjogMTM5OH0sIHsiZGF0ZSI6ICIyMDI0LTEwLTE0IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTI1Mn0sIHsiZGF0ZSI6ICIyMDI0LTEwLTI0IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAwMiwgInJpbmdfbWVhbl92aXMiOiAxMzEzfSwgeyJkYXRlIjogIjIwMjQtMTAtMjkiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMDE0LCAicmluZ19tZWFuX3ZpcyI6IDEyNzR9LCB7ImRhdGUiOiAiMjAyNC0xMi0wOCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDExNzd9LCB7ImRhdGUiOiAiMjAyNC0xMi0wOCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wMDIsICJyaW5nX21lYW5fdmlzIjogMTE4N30sIHsiZGF0ZSI6ICIyMDI0LTEyLTEzIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTIxNX0sIHsiZGF0ZSI6ICIyMDI0LTEyLTI4IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTE2M30sIHsiZGF0ZSI6ICIyMDI1LTAxLTA3IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAsICJyaW5nX21lYW5fdmlzIjogMTE1NH0sIHsiZGF0ZSI6ICIyMDI1LTAxLTIyIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAxLCAicmluZ19tZWFuX3ZpcyI6IDEyNTB9LCB7ImRhdGUiOiAiMjAyNS0wMi0wNiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDExODV9LCB7ImRhdGUiOiAiMjAyNS0wMi0yMSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyNjB9LCB7ImRhdGUiOiAiMjAyNS0wMi0yNiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wLCAicmluZ19tZWFuX3ZpcyI6IDEyODV9LCB7ImRhdGUiOiAiMjAyNS0wMy0wMyIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wMDUsICJyaW5nX21lYW5fdmlzIjogMTI3N30sIHsiZGF0ZSI6ICIyMDI1LTA0LTA5IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAyMSwgInJpbmdfbWVhbl92aXMiOiAxMDk0fSwgeyJkYXRlIjogIjIwMjUtMDQtMTciLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMDYsICJyaW5nX21lYW5fdmlzIjogMTM3MH0sIHsiZGF0ZSI6ICIyMDI1LTA3LTI2IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjA1NSwgInJpbmdfbWVhbl92aXMiOiAxNTgxfSwgeyJkYXRlIjogIjIwMjUtMDktMDYiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMDIxLCAicmluZ19tZWFuX3ZpcyI6IDE3OTd9LCB7ImRhdGUiOiAiMjAyNS0wOS0xNCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC4wOTgsICJyaW5nX21lYW5fdmlzIjogMTkzOH0sIHsiZGF0ZSI6ICIyMDI1LTA5LTE2IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjAwMiwgInJpbmdfbWVhbl92aXMiOiAxNzk0fSwgeyJkYXRlIjogIjIwMjUtMDktMjQiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuMjA1LCAicmluZ19tZWFuX3ZpcyI6IDIxMDN9LCB7ImRhdGUiOiAiMjAyNS0xMS0xNSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC40NDYsICJyaW5nX21lYW5fdmlzIjogMjM5Nn0sIHsiZGF0ZSI6ICIyMDI1LTEyLTA1IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjQxNSwgInJpbmdfbWVhbl92aXMiOiAyMzY1fSwgeyJkYXRlIjogIjIwMjUtMTItMjUiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuNDUzLCAicmluZ19tZWFuX3ZpcyI6IDIzNDh9LCB7ImRhdGUiOiAiMjAyNi0wMS0wMiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC40MDEsICJyaW5nX21lYW5fdmlzIjogMjI4NX0sIHsiZGF0ZSI6ICIyMDI2LTAxLTA0IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjQ1MSwgInJpbmdfbWVhbl92aXMiOiAyMzg5fSwgeyJkYXRlIjogIjIwMjYtMDEtMDciLCAicmluZ19icmlnaHRfZnJhYyI6IDAuNjA2LCAicmluZ19tZWFuX3ZpcyI6IDI3MTd9LCB7ImRhdGUiOiAiMjAyNi0wMi0xMSIsICJyaW5nX2JyaWdodF9mcmFjIjogMC44NjIsICJyaW5nX21lYW5fdmlzIjogNDgyMn0sIHsiZGF0ZSI6ICIyMDI2LTA4LTA1IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjg5LCAicmluZ19tZWFuX3ZpcyI6IDQ5MzR9LCB7ImRhdGUiOiAiMjAyNi0wOC0xMiIsICJyaW5nX2JyaWdodF9mcmFjIjogMC44ODUsICJyaW5nX21lYW5fdmlzIjogNTEyMn0sIHsiZGF0ZSI6ICIyMDI2LTA4LTI1IiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjkwNywgInJpbmdfbWVhbl92aXMiOiA1NDI3fSwgeyJkYXRlIjogIjIwMjYtMDktMDQiLCAicmluZ19icmlnaHRfZnJhYyI6IDAuOTE5LCAicmluZ19tZWFuX3ZpcyI6IDU4NTJ9LCB7ImRhdGUiOiAiMjAyNi0wOS0yNCIsICJyaW5nX2JyaWdodF9mcmFjIjogMC45MDcsICJyaW5nX21lYW5fdmlzIjogNjA3MH0sIHsiZGF0ZSI6ICIyMDI2LTEwLTAxIiwgInJpbmdfYnJpZ2h0X2ZyYWMiOiAwLjkwNSwgInJpbmdfbWVhbl92aXMiOiA0OTgwfV19"))
    print("manifest.json written from the notebook copy")
print("site_pad_y", json.load(open(_manifest))["features"]["site_pad_y"])


In [ ]:
# The inline gpu_pre.py of the GPU test notebook (same as blender/kaggle/gpu_pre.py in the repo).
GPU_PRE = f"{WORK}/gpu_pre.py"
open(GPU_PRE, "w").write('''prefs = bpy.context.preferences.addons["cycles"].preferences
chosen = None
for kind in ("OPTIX", "CUDA"):
    try:
        prefs.compute_device_type = kind
    except TypeError:
        continue
    prefs.get_devices()
    if any(d.type == kind for d in prefs.devices):
        chosen = kind
        break
if chosen is None:
    raise SystemExit("gpu_pre.py: no OptiX or CUDA device found")
for d in prefs.devices:
    d.use = d.type == chosen
s.cycles.device = "GPU"
print("gpu_pre.py:", chosen, [d.name for d in prefs.devices if d.use])
''')
print("setup", round(time.time() - t_setup), "s")

In [ ]:
# Overlay of blender/web_hero/render_panorama.py at c798e62 (out= paths outside the repo). Older packs have the version that fails after the save.
open(f"{PACK}/blender/web_hero/render_panorama.py", "wb").write(base64.b64decode("IiIiV2ViIGhlcm8gc3RlcCAyOiAzNjAtZGVncmVlIGVxdWlyZWN0YW5ndWxhciBIRFIgZW52aXJvbm1lbnQgZm9yIHRoZSB0aHJlZS5qcyBhcHAuCgpSdW4gZnJvbSB0aGUgcmVwbyByb290OgogIH4vLnZlbnZzL2JweS9iaW4vcHl0aG9uIGJsZW5kZXIvd2ViX2hlcm8vcmVuZGVyX3Bhbm9yYW1hLnB5IFtzYW1wbGVzXSBbd2lkdGhdIFtwcmVzZXQ9bW9ybmluZ19nb2xkZW58ZXZlbmluZ19nb2xkZW5dCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBbbmVhcj08bT5dIFtvdXQ9PGRpcj5dCkNhbWVyYSAxNTAgbSBhYm92ZSB0aGUgcGl0Y2ggY2VudHJlLCBDUFUgQ3ljbGVzLCB0aGUgYXBwcm92ZWQgZ29sZGVuIHN1biBhbmQgc2t5IChtb3JuaW5nX2dvbGRlbiAwNzozMCBieSBkZWZhdWx0LApldmVuaW5nX2dvbGRlbiAxNzozMCkuIFRoZSBzdGFkaXVtIChwbGludGgsIHNoZWxsLCBmYWNhZGUgZGV0YWlsLCB0aWVycywgYm93bCBhbmQgcm9vZiBkZXRhaWwsIHBpdGNoLCBuaWdodCBmaXh0dXJlcykKaXMgaGlkZGVuLCBzbyB0aGUgYXBwIGNhbiBwbGFjZSB0aGUgR0xCIGluc2lkZSB0aGUgcGFub3JhbWEuIG5lYXI9PG0+IGFsc28gcmVtb3ZlcyBldmVyeXRoaW5nIHdpdGhpbiA8bT4gb2YgdGhlCnN0YWRpdW0gY2VudHJlIGluIHBsYW4gKG9iamVjdHMgd2hvbGx5IGluc2lkZSBhcmUgaGlkZGVuOyBvYmplY3RzIGNyb3NzaW5nIHRoZSBjaXJjbGUgZ2V0IGEgZ2VvbWV0cnktbm9kZXMgY3V0IG9mCnRoZWlyIHBvaW50cyBhbmQgaW5zdGFuY2VzKSwgc28gdGhlIHBhbm9yYW1hIGNhcnJpZXMgb25seSB0aGUgZmFyIGNvbnRleHQgKGhpbGxzLCByYWNlY291cnNlLCBjaXR5LCBDQkQpIGFuZCB0aGUgR0xCcwpjYXJyeSB0aGUgcmVzdC4gV3JpdGVzIFJhZGlhbmNlIEhEUiAobGluZWFyLCBubyBleHBvc3VyZSBvciB0b25lIG1hcHBpbmcpIGF0IGZ1bGwgd2lkdGggYW5kIDIwNDggcHgsIHBsdXMgYSBsb2cuCkRlZmF1bHQgb3V0cHV0OiBhcHAvcHVibGljL2Vudi90YWxhbnRhXzA3MzUuaGRyICh0aGUgb3JpZ2luYWwgc2luZ2xlIG1vcm5pbmcgcGFub3JhbWEpLgoiIiIKaW1wb3J0IG1hdGgsIHN5cywgdGltZQpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKc3lzLnBhdGguaW5zZXJ0KDAsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkucGFyZW50KSkKZnJvbSBjb21tb24gaW1wb3J0IFJPT1QsIE9VVCwgc2V0dXBfbW9ybmluZywgc3VuX3ZlY3Rvcl9hcHAsIHdyaXRlX2pzb24KCiMgc2NyaXB0IGFyZ3VtZW50cyBvbmx5OiB1bmRlciB0aGUgYmxlbmRlciBiaW5hcnkgdGhleSBmb2xsb3cgIi0tIiAoQmxlbmRlcidzIG93biwgZS5nLiAtLXB5dGhvbi1leGl0LWNvZGUgMSwgY29tZSBmaXJzdCkKQVJHViA9IHN5cy5hcmd2W3N5cy5hcmd2LmluZGV4KCItLSIpICsgMTpdIGlmICItLSIgaW4gc3lzLmFyZ3YgZWxzZSBzeXMuYXJndlsxOl0KYXJncyA9IFthIGZvciBhIGluIEFSR1YgaWYgYS5pc2RpZ2l0KCldCktXID0gZGljdChhLnNwbGl0KCI9IiwgMSkgZm9yIGEgaW4gQVJHViBpZiAiPSIgaW4gYSkKU0FNUExFUyA9IGludChhcmdzWzBdKSBpZiBhcmdzIGVsc2UgMTI4ClcgPSBpbnQoYXJnc1sxXSkgaWYgbGVuKGFyZ3MpID4gMSBlbHNlIDQwOTYKUFJFU0VUID0gS1cuZ2V0KCJwcmVzZXQiLCAibW9ybmluZ19nb2xkZW4iKQpORUFSID0gZmxvYXQoS1dbIm5lYXIiXSkgaWYgIm5lYXIiIGluIEtXIGVsc2UgTm9uZQpnID0gc2V0dXBfbW9ybmluZyhwcmVzZXQ9UFJFU0VUKQppbXBvcnQgYnB5CgpzID0gYnB5LmNvbnRleHQuc2NlbmUKSElERSA9ICgicGxpbnRoIiwgInNoZWxsIiwgImZhY2FkZV9kZXRhaWwiLCAidGllcnMiLCAiYm93bF9kZXRhaWwiLCAicm9vZl9kZXRhaWwiLCAicGl0Y2giLCAibmlnaHRfbGlnaHRzIiwgImJvd2xfcHJvdG9zIikKaGlkZGVuID0gW10KZm9yIGMgaW4gYnB5LmRhdGEuY29sbGVjdGlvbnM6CiAgICBpZiBjLm5hbWUgaW4gSElERToKICAgICAgICBmb3IgbyBpbiBjLmFsbF9vYmplY3RzOgogICAgICAgICAgICBpZiBvIGlzIG5vdCBOb25lIGFuZCBub3Qgby5oaWRlX3JlbmRlcjogby5oaWRlX3JlbmRlciA9IFRydWU7IGhpZGRlbi5hcHBlbmQoby5uYW1lKQoKCmRlZiBuZWFyX2N1dChyYWRpdXMpOgogICAgIiIiSGlkZSBvYmplY3RzIHdpdGhpbiByYWRpdXMgKHBsYW4pIG9mIHRoZSBvcmlnaW4uIE1lc2hlcyB0aGF0IGNyb3NzIHRoZSBjaXJjbGUgZ2V0IHRoZWlyIG93biBjb3B5IG9mIHRoZSBtZXNoCiAgICB3aXRoIGV2ZXJ5IHZlcnRleCBpbnNpZGUgaXQgZGVsZXRlZCAoYm1lc2g7IGZhY2VzIGdvIHdpdGggdGhlaXIgdmVydGljZXMpLiBUaGF0IGFsc28gY3V0cyBnZW9tZXRyeS1ub2RlcyBwb2ludAogICAgc2NhdHRlcnMgKHRoZWlyIGluc3RhbmNlcyBzaXQgb24gbWVzaCBwb2ludHMsIGF0dHJpYnV0ZXMga2VwdCkuIEEgZ2VvbWV0cnktbm9kZXMgY3V0IGV2YWx1YXRlZCByaWdodCBpbiB0aGUKICAgIHZpZXdwb3J0IGJ1dCByZW1vdmVkIHRoZSB3aG9sZSBtZXNoIGF0IHJlbmRlciB0aW1lLCBzbyB0aGUgY3V0IGlzIG1hZGUgb24gdGhlIGRhdGEuIEN1cnZlcyBhcmUgbGVmdCB3aG9sZS4iIiIKICAgIGltcG9ydCBibWVzaAogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBmcm9tIG1hdGh1dGlscyBpbXBvcnQgVmVjdG9yCiAgICBuX2hpZGUgPSBuX2N1dCA9IDAKICAgIGZvciBvIGluIHMub2JqZWN0czoKICAgICAgICBpZiBvLmhpZGVfcmVuZGVyIG9yIG8udHlwZSBub3QgaW4gKCJNRVNIIiwgIkNVUlZFIiwgIkVNUFRZIik6IGNvbnRpbnVlCiAgICAgICAgaWYgby50eXBlID09ICJFTVBUWSI6CiAgICAgICAgICAgIGlmIG8uaW5zdGFuY2VfdHlwZSA9PSAiQ09MTEVDVElPTiIgYW5kIFZlY3RvcihvLm1hdHJpeF93b3JsZC50cmFuc2xhdGlvbls6Ml0pLmxlbmd0aCA8IHJhZGl1czogby5oaWRlX3JlbmRlciA9IFRydWU7IG5faGlkZSArPSAxCiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgYmIgPSBbby5tYXRyaXhfd29ybGQgQCBWZWN0b3IoYykgZm9yIGMgaW4gby5ib3VuZF9ib3hdCiAgICAgICAgaWYgbWF4KFZlY3Rvcih2WzoyXSkubGVuZ3RoIGZvciB2IGluIGJiKSA8IHJhZGl1czoKICAgICAgICAgICAgby5oaWRlX3JlbmRlciA9IFRydWU7IG5faGlkZSArPSAxOyBjb250aW51ZQogICAgICAgIGlmIG8udHlwZSAhPSAiTUVTSCI6IGNvbnRpbnVlCiAgICAgICAgeHMsIHlzID0gW3YueCBmb3IgdiBpbiBiYl0sIFt2LnkgZm9yIHYgaW4gYmJdCiAgICAgICAgaWYgVmVjdG9yKChtaW4obWF4KDAuMCwgbWluKHhzKSksIG1heCh4cykpLCBtaW4obWF4KDAuMCwgbWluKHlzKSksIG1heCh5cykpKSkubGVuZ3RoID49IHJhZGl1czogY29udGludWUKICAgICAgICBvLmRhdGEgPSBvLmRhdGEuY29weSgpCiAgICAgICAgY28gPSBucC5lbXB0eShsZW4oby5kYXRhLnZlcnRpY2VzKSAqIDMpOyBvLmRhdGEudmVydGljZXMuZm9yZWFjaF9nZXQoImNvIiwgY28pOyBjbyA9IGNvLnJlc2hhcGUoLTEsIDMpCiAgICAgICAgTSA9IG5wLmFycmF5KG8ubWF0cml4X3dvcmxkKTsgdyA9IGNvIEAgTVs6MywgOjNdLlQgKyBNWzozLCAzXQogICAgICAgIGluc2lkZSA9IG5wLmh5cG90KHdbOiwgMF0sIHdbOiwgMV0pIDwgcmFkaXVzCiAgICAgICAgaWYgbm90IGluc2lkZS5hbnkoKTogY29udGludWUKICAgICAgICBibSA9IGJtZXNoLm5ldygpOyBibS5mcm9tX21lc2goby5kYXRhKTsgYm0udmVydHMuZW5zdXJlX2xvb2t1cF90YWJsZSgpCiAgICAgICAgYm1lc2gub3BzLmRlbGV0ZShibSwgZ2VvbT1bYm0udmVydHNbaV0gZm9yIGkgaW4gbnAubm9uemVybyhpbnNpZGUpWzBdXSwgY29udGV4dD0iVkVSVFMiKQogICAgICAgIGJtLnRvX21lc2goby5kYXRhKTsgYm0uZnJlZSgpOyBvLmRhdGEudXBkYXRlKCk7IG5fY3V0ICs9IDEKICAgIHJldHVybiBuX2hpZGUsIG5fY3V0CgoKY3V0ID0gbmVhcl9jdXQoTkVBUikgaWYgTkVBUiBlbHNlIE5vbmUKY2FtX2QgPSBicHkuZGF0YS5jYW1lcmFzLm5ldygicGFubyIpOyBjYW1fZC50eXBlID0gIlBBTk8iCmNhbV9kLmNsaXBfc3RhcnQsIGNhbV9kLmNsaXBfZW5kID0gMS4wLCAxMDAwMDAuMCAgIyBhIG5ldyBjYW1lcmEgY2xpcHMgYXQgMSBrbSwgd2hpY2ggY3V0IHRoZSBmYXIgbGFuZCBvdXQgb2YgdGhlIHBhbm9yYW1hCnRyeTogY2FtX2QucGFub3JhbWFfdHlwZSA9ICJFUVVJUkVDVEFOR1VMQVIiCmV4Y2VwdCBBdHRyaWJ1dGVFcnJvcjogY2FtX2QuY3ljbGVzLnBhbm9yYW1hX3R5cGUgPSAiRVFVSVJFQ1RBTkdVTEFSIgpjYW0gPSBicHkuZGF0YS5vYmplY3RzLm5ldygicGFubyIsIGNhbV9kKTsgcy5jb2xsZWN0aW9uLm9iamVjdHMubGluayhjYW0pCmltcG9ydCBqc29uClBBRCA9IGpzb24ubG9hZHMoKFJPT1QgLyAiYXBwL3B1YmxpYy9kYXRhL21hbmlmZXN0Lmpzb24iKS5yZWFkX3RleHQoKSlbImZlYXR1cmVzIl1bInNpdGVfcGFkX3kiXQpjYW0ubG9jYXRpb24gPSAoMC4wLCAwLjAsIFBBRCArIDE1MC4wKQojIEJsZW5kZXIgZXF1aXJlY3Q6IGltYWdlIGNlbnRyZSBsb29rcyBhbG9uZyB0aGUgY2FtZXJhJ3MgLVo7IHJvdGF0ZSBzbyB0aGUgY2VudHJlIGxvb2tzIG5vcnRoICgrWSkgYW5kIHVwIGlzICtaLgpjYW0ucm90YXRpb25fZXVsZXIgPSAobWF0aC5yYWRpYW5zKDkwKSwgMC4wLCAwLjApCnMuY2FtZXJhID0gY2FtCnMucmVuZGVyLnJlc29sdXRpb25feCwgcy5yZW5kZXIucmVzb2x1dGlvbl95LCBzLnJlbmRlci5yZXNvbHV0aW9uX3BlcmNlbnRhZ2UgPSBXLCBXIC8vIDIsIDEwMApjID0gcy5jeWNsZXM7IGMuc2FtcGxlcyA9IFNBTVBMRVM7IGMudXNlX2FkYXB0aXZlX3NhbXBsaW5nID0gVHJ1ZTsgYy51c2VfZGVub2lzaW5nID0gVHJ1ZTsgYy5kZW5vaXNlciA9ICJPUEVOSU1BR0VERU5PSVNFIgppZiBfX2ltcG9ydF9fKCJvcyIpLmVudmlyb24uZ2V0KCJUQUxBTlRBX1BSRSIpOiAgIyBHUFUgcnVucyAoS2FnZ2xlKTogYmxlbmRlci9rYWdnbGUvZ3B1X3ByZS5weSBzZXRzIHRoZSBDeWNsZXMgZGV2aWNlIHRvIEdQVQogICAgX3ByZSA9IF9faW1wb3J0X18oIm9zIikuZW52aXJvblsiVEFMQU5UQV9QUkUiXTsgZXhlYyhjb21waWxlKG9wZW4oX3ByZSkucmVhZCgpLCBfcHJlLCAiZXhlYyIpLCB7ImJweSI6IGJweSwgInMiOiBzLCAiUk9PVCI6IFJPT1R9KQpzLnZpZXdfc2V0dGluZ3Mudmlld190cmFuc2Zvcm0gPSAiU3RhbmRhcmQiOyBzLnZpZXdfc2V0dGluZ3MuZXhwb3N1cmUgPSAwLjA7IHMudmlld19zZXR0aW5ncy5sb29rID0gIk5vbmUiCnMucmVuZGVyLmltYWdlX3NldHRpbmdzLmZpbGVfZm9ybWF0ID0gIkhEUiIKaWYgIm91dCIgaW4gS1c6ICAjIGJ1bmRsZTogdGhlIGZ1bGwgcGFub3JhbWEgYXMgaGFsZi1mbG9hdCBPcGVuRVhSIChEV0FCKSwgZmFyIHNtYWxsZXIgdGhhbiBSYWRpYW5jZSBIRFIgYXQgOEsKICAgIHMucmVuZGVyLmltYWdlX3NldHRpbmdzLmZpbGVfZm9ybWF0ID0gIk9QRU5fRVhSIjsgcy5yZW5kZXIuaW1hZ2Vfc2V0dGluZ3MuY29sb3JfZGVwdGggPSAiMTYiOyBzLnJlbmRlci5pbWFnZV9zZXR0aW5ncy5leHJfY29kZWMgPSAiRFdBQiIKaGhtbSA9IGdbInN0Il1bInRpbWVfZWF0Il0ucmVwbGFjZSgiOiIsICIiKQppZiAib3V0IiBpbiBLVzoKICAgIGVudiA9IFJPT1QgLyBLV1sib3V0Il07IHN0ZW0gPSBmInRhbGFudGFfcGFub197aGhtbX0iCmVsc2U6CiAgICBlbnYgPSBST09UIC8gImFwcC9wdWJsaWMvZW52Ijsgc3RlbSA9ICJ0YWxhbnRhXzA3MzUiCmVudi5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCmZ1bGwgPSBlbnYgLyAoZiJ7c3RlbX0uZXhyIiBpZiAib3V0IiBpbiBLVyBlbHNlIGYie3N0ZW19LmhkciIpCnQwID0gdGltZS50aW1lKCk7IHMucmVuZGVyLmZpbGVwYXRoID0gc3RyKGZ1bGwpOyBicHkub3BzLnJlbmRlci5yZW5kZXIod3JpdGVfc3RpbGw9VHJ1ZSk7IHNlYyA9IHRpbWUudGltZSgpIC0gdDAKcHJldl9kaXIgPSBPVVQgLyAicGFub19wcmV2aWV3IiBpZiAib3V0IiBpbiBLVyBlbHNlIGVudjsgcHJldl9kaXIubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQojIHByZXZpZXcgZm9yIHJldmlldyBvbmx5OiB0aGUgc2FtZSByZW5kZXIgdGhyb3VnaCBBZ1ggYXQgdGhlIHByZXNldCdzIGV4cG9zdXJlLCAyMDQ4IHB4IEpQRUcKcnIgPSBicHkuZGF0YS5pbWFnZXNbIlJlbmRlciBSZXN1bHQiXTsgcy52aWV3X3NldHRpbmdzLnZpZXdfdHJhbnNmb3JtID0gIkFnWCI7IHMudmlld19zZXR0aW5ncy5leHBvc3VyZSA9IC0xLjAgaWYgUFJFU0VUID09ICJtb3JuaW5nX2dvbGRlbiIgZWxzZSAtMS43ICAjIGFwcHJvdmVkIHBhc3MgMSBsb29rcwpzLnJlbmRlci5pbWFnZV9zZXR0aW5ncy5maWxlX2Zvcm1hdCA9ICJKUEVHIjsgcy5yZW5kZXIuaW1hZ2Vfc2V0dGluZ3MucXVhbGl0eSA9IDg4CnJyLnNhdmVfcmVuZGVyKHN0cihwcmV2X2RpciAvIGYie3N0ZW19X3ByZXZpZXcuanBnIiksIHNjZW5lPXMpCnB2ID0gYnB5LmRhdGEuaW1hZ2VzLmxvYWQoc3RyKHByZXZfZGlyIC8gZiJ7c3RlbX1fcHJldmlldy5qcGciKSk7IHB2LnNjYWxlKDIwNDgsIDEwMjQpOyBwdi5zYXZlKCkKIyAyMDQ4IHB4IHZlcnNpb24gZnJvbSB0aGUgZnVsbCByZW5kZXIgKGJveCBmaWx0ZXIpLCBzbyBib3RoIGZpbGVzIHNob3cgdGhlIHNhbWUgbGlnaHQKaW0gPSBicHkuZGF0YS5pbWFnZXMubG9hZChzdHIoZnVsbCkpOyBpbS5zY2FsZSgyMDQ4LCAxMDI0KQppbS5maWxlcGF0aF9yYXcgPSBzdHIoZW52IC8gZiJ7c3RlbX1fMjA0OC5oZHIiKTsgaW0uZmlsZV9mb3JtYXQgPSAiSERSIjsgaW0uc2F2ZSgpCmVsLCBheiA9IGdbImVsIl0sIGdbImF6Il0KIyBvdXQ9IGNhbiBiZSBvdXRzaWRlIHRoZSByZXBvIChLYWdnbGUgd3JpdGVzIHRvIC9rYWdnbGUvd29ya2luZyksIHNvIHJlY29yZCBzdWNoIHBhdGhzIGFzIHRoZXkgYXJlLgpyZWwgPSBsYW1iZGEgcDogc3RyKHAucmVsYXRpdmVfdG8oUk9PVCkpIGlmIHAuaXNfcmVsYXRpdmVfdG8oUk9PVCkgZWxzZSBzdHIocCkKd3JpdGVfanNvbigoZW52IGlmICJvdXQiIGluIEtXIGVsc2UgT1VUKSAvIGYicGFub3JhbWFfbG9neydfJyArIGhobW0gaWYgJ291dCcgaW4gS1cgZWxzZSAnJ30uanNvbiIsIHsKICAgICJmaWxlIjogcmVsKGZ1bGwpLCAiZmlsZV8yMDQ4IjogcmVsKGVudiAvIGYie3N0ZW19XzIwNDguaGRyIiksCiAgICAicHJlc2V0IjogUFJFU0VULCAicmVzb2x1dGlvbiI6IFtXLCBXIC8vIDJdLCAic2FtcGxlcyI6IFNBTVBMRVMsICJzZWNvbmRzIjogcm91bmQoc2VjLCAxKSwKICAgICJuZWFyX2N1dF9tIjogTkVBUiwgIm5lYXJfY3V0X29iamVjdHMiOiB7ImhpZGRlbiI6IGN1dFswXSwgImN1dCI6IGN1dFsxXX0gaWYgY3V0IGVsc2UgTm9uZSwKICAgICJjYW1lcmFfYXBwIjogWzAuMCwgUEFEICsgMTUwLjAsIDAuMF0sICJjZW50cmVfb2ZfaW1hZ2VfbG9va3MiOiAibm9ydGggKC16IGFwcCkiLCAicHJvamVjdGlvbiI6ICJlcXVpcmVjdGFuZ3VsYXIiLAogICAgInN1bl9lbGV2YXRpb25fZGVnIjogcm91bmQoZWwsIDIpLCAic3VuX2F6aW11dGhfZGVnIjogcm91bmQoYXosIDIpLCAic3VuX2Rpcl9hcHAiOiBzdW5fdmVjdG9yX2FwcChlbCwgYXopLAogICAgImNvbG91ciI6ICJsaW5lYXIgUmVjLjcwOSwgbm8gZXhwb3N1cmUsIG5vIHRvbmUgbWFwcGluZyAocmVuZGVyX2N5Y2xlcyB1c2VzIEFnWCBhdCAtMSBFViBmb3IgdGhlIG1vcm5pbmcgbG9vaykiLAogICAgImhpZGRlbiI6IHNvcnRlZChzZXQoaGlkZGVuKSl9KQpwcmludCh7InNlY29uZHMiOiByb3VuZChzZWMsIDEpLCAibmVhcl9jdXQiOiBjdXR9KQo="))
print("render_panorama.py overlay", os.path.getsize(f"{PACK}/blender/web_hero/render_panorama.py"), "bytes")


In [ ]:
# The panoramas, one Blender process for each time
import hashlib
env = dict(os.environ, TALANTA_SITE_BLEND=f"{PACK}/blender/out/frozen_FL.blend", TALANTA_PRE=GPU_PRE, PYTHONUNBUFFERED="1")
rec = {}
for HHMM in TIMES:
    OUT = f"/kaggle/working/env/{HHMM}"; os.makedirs(OUT, exist_ok=True)
    cmd = [BLENDER, "-b", "--factory-startup", "--python-exit-code", "1", "--python", f"{PACK}/blender/web_hero/render_panorama.py", "--",
           str(SAMPLES), "8192", f"preset={PRESETS[HHMM]}", "near=800", f"out={OUT}"]
    t0 = time.time(); r = subprocess.run(cmd, cwd=PACK, env=env, capture_output=True, text=True)
    open(f"{OUT}/pano_run.log", "w").write(r.stdout[-100000:] + r.stderr[-20000:])
    print(HHMM, "panorama", round((time.time() - t0) / 60, 1), "min, rc", r.returncode); assert r.returncode == 0, r.stderr[-3000:]
    rec[HHMM] = {"preset": PRESETS[HHMM], "minutes": round((time.time() - t0) / 60, 2),
                 "sha256": {f: hashlib.sha256(open(f"{OUT}/{f}", "rb").read()).hexdigest() for f in sorted(os.listdir(OUT)) if not f.endswith(".log")}}
json.dump({"samples": SAMPLES, "times": rec}, open("/kaggle/working/env/run_record.json", "w"), indent=1)
print(json.dumps(rec, indent=1)[:3000])


In [ ]:
# Keep only the panoramas in the version output
shutil.rmtree(WORK, ignore_errors=True)
!du -sh /kaggle/working/env/*
